In [2]:
import pandas as pd

In [8]:
df = pd.read_csv("../customer_shopping_behavior_dataset.csv")

In [9]:
df.sample(10)

df.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [ ]:
# Fills missing ratings using the median of their specific product category 
# .groupby() groups the rows; .transform() applies the custom lambda fix 
# and stretches the results back out to perfectly match the original row count.
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [ ]:
df.isnull().sum() # cleaned Null Values By replacing them with median opf thier category

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

In [ ]:
#Changing column names to snake case for better usability
# .str acts as a string accessor/toolbox. It allows you to perform 
# standard string operations (like .lower() and .replace()) on every 
# item simultaneously, WITHOUT converting the pandas Index object 
# into a standard Python list of strings.

df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(" ","_")
df = df.rename(columns = {'purchase_amount_(usd)':'purchase_amount'})

In [ ]:
#Creating Age Group Labels
# pd.qcut automatically SORTS the data internally and divides it into 
# 4 equal-sized groups based on row count (e.g., exactly 5 rows each out of 20).
# It calculates the percentage limits automatically without converting or changing 
# your original row positions, assigning your sequential labels from youngest to oldest.
age_category = ['Young Adult','Adult','Middle-aged','Senior']

df['age_group'] = pd.qcut(df['age'],q = 4,labels = age_category)
df[['age','age_group']].tail(10)

,age,age_group
3880,42,Adult
3881,56,Middle-aged
3882,35,Adult
3883,34,Adult
3884,47,Middle-aged
3885,49,Middle-aged
3886,37,Adult
3887,40,Adult
3888,45,Middle-aged
3889,57,Middle-aged


In [ ]:
#replacing str values with logical numerical values
#create column purchase_frequency_days
# .map() acts as a direct translation dictionary for column values.
# It checks every row in the 'frequency_of_purchases' column, looks up 
# the text string (like 'Fortnightly') inside the frequency_mapping dictionary, 
# and replaces it with the corresponding numerical value (like 14) in the new column.
frequency_mapping = {
    'Fortnightly': 14,
    'Weekly': 7,
    'Bi-Weekly': 14,
    'Monthly': 30,
    'Quarterly': 90,
    'Annually': 365,
    'Every 3 Months': 90
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)
df[['purchase_frequency_days','frequency_of_purchases']].head(10)
df.drop('frequency_of_purchases',axis = 1)

,purchase_frequency_days,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually
5,7,Weekly
6,90,Quarterly
7,7,Weekly
8,365,Annually
9,90,Quarterly


In [38]:
#CHECKING : If EveryDiscount Was Given After Applying PromoCode Or Not

(df['promo_code_used'] == df['discount_applied']).all()

df = df.drop('promo_code_used',axis = 1)
df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='str')

In [42]:
pip install pyodbc sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [ ]:
#To connect Python Jupyter With SQL Server Remeber This code may vary according to diff DBs like for SQL Server We install 'pyodbc sqlalchemy' 
#Where as for Postgress We use 'psycopg2-binary sqlalchemy' package

from sqlalchemy import create_engine
import urllib


# ####################################################################
# Step 1: Connect to SQL Server
# ####################################################################

# REPLACE THESE WITH THE INFOS YOU FIND IN THE STEPS BELOW:
server = "DESKTOP-0BKD8VE\SQLEXPRESS"       
database = "customer_behavior"    # The database you just created

# Use this block for Windows Authentication (Standard Industry Default for local work)
params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
)

# Initialize the engine
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# ####################################################################
# Step 2: Load DataFrame into SQL Server
# ####################################################################
table_name = "customer"           
df.to_sql(table_name, engine, if_exists="replace", index=False)

print(f"Data successfully loaded into table '{table_name}' in database '{database}'.")

<>:10: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
<>:10: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
C:\Users\DELL\AppData\Local\Temp\ipykernel_8484\3627144708.py:10: SyntaxWarning: "\S" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\S"? A raw string is also an option.
  server = "DESKTOP-0BKD8VE\SQLEXPRESS"


Data successfully loaded into table 'customer' in database 'customer_behavior'.
